In [ ]:
import h5py
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.mixture import GaussianMixture
from scipy.special import erf
from scipy.optimize import least_squares
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 -- registers the '3d' projection
from matplotlib.backends.backend_pdf import PdfPages
from pathlib import Path
import time

In [ ]:
base_dir      = Path("/Users/angelaliu/Library/CloudStorage/OneDrive-MassachusettsInstituteofTechnology/Fracture Flow UROP")
# transport_dir = base_dir / "Updated Code" / "transport narrowed param ranges"
transport_dir = base_dir / "Updated Code" / "transport"
mat_path = transport_dir / "all_trial_bkt_data.mat"

In [ ]:
# Reused from the sibling percentile-BKT notebooks -- keep in sync if those
# notebooks' loader changes.
def load_all_trial_bkt_data(mat_path, ps_indices=None):
    param_sets = {}
    with h5py.File(mat_path, 'r') as f:
        weights = f['weights'][()].T  # (num_param_sets, 3)
        atbd = f['all_trial_bkt_data']  # (1, N) object refs
        indices = ps_indices if ps_indices is not None else range(atbd.shape[1])
        for i in indices:
            ps_grp = f[atbd[0, i]]
            if 'p0' in ps_grp:
                p0 = float(ps_grp['p0'][()].flat[0])
            else:
                # The OG "transport" dataset (original, non-narrowed param
                # ranges -- cell-1 points here) stores connectivity as
                # connected_fraction instead of p0; p0 (disconnected
                # probability) is just its complement. Everything downstream
                # of this loader still only ever sees p0.
                p0 = 1.0 - float(ps_grp['connected_fraction'][()].flat[0])
            td = ps_grp['trial_data']
            trial_data = []
            for j in range(td.shape[1]):
                arr = f[td[0, j]][()]
                trial_data.append(arr.flatten() if arr.dtype == np.float64 else np.array([], dtype=np.float64))
            param_sets[i] = {'p0': p0, 'trial_data': trial_data}
    return param_sets, weights


def recover_theta(w):
    """Raw [0,1] weight -> physical theta. OG param ranges (matches
    Updated Code/transport, per info_outputs_varying_parameters.m -- NOT the
    narrowed-range bounds used by the "transport narrowed param ranges"
    dataset elsewhere in this project): disc in [0.1, 5] (was [1, 2.1]
    narrowed), length_ratio in [0.02, 0.5] (was [0.3, 0.5] narrowed); b is
    unchanged, [0.01e-3, 0.3e-3] in both."""
    disc = 5*w[0] + 0.1*(1 - w[0])            # [0.1, 5]
    b    = 0.01e-3*w[1] + 0.3e-3*(1 - w[1])   # [0.01e-3, 0.3e-3]
    lr   = 0.02*w[2] + 0.5*(1 - w[2])         # [0.02, 0.5]
    return disc, lr, b


def quantile_bkt(bkt, p, Np):
    """pth-percentile breakthrough time for one trial from its raw particle
    arrival times (order statistic), or None if undefined. Copied from
    [per p] distrib of pth percentile bkt.ipynb."""
    k = int(np.floor(p * Np))
    if k < len(bkt):
        return bkt[k]
    return None


def logit(p, eps=1e-4):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))

In [ ]:
# Same connectivity filter as the sibling notebooks.
P0_THRESHOLD = 0.9
Np = 1000

with h5py.File(mat_path, 'r') as f:
    n_param_sets = f['all_trial_bkt_data'].shape[1]

all_param_sets, weights = load_all_trial_bkt_data(mat_path, range(n_param_sets))
USABLE_PS = sorted(i for i, ps in all_param_sets.items() if ps['p0'] < P0_THRESHOLD)
ps_by_p0 = sorted(USABLE_PS, key=lambda i: all_param_sets[i]['p0'])
print(f"{len(USABLE_PS)} / {n_param_sets} param sets usable (p0 < {P0_THRESHOLD})")

In [ ]:
# --- New approach: learn the joint CDF surface F(theta, p, log t) directly,
# instead of fitting a parametric family (2-Gaussian mixture / skew-logistic)
# at each (theta,p) and training a network to predict that family's
# coefficients. Motivated by the joint-surface plots in the sibling notebook
# ("[per p] joint CDF surface F(theta,p,logt).ipynb"), which show the whole
# family of percentile-BKT CDFs forms one smooth, coherent surface in
# (p, log t) rather than a set of unrelated curves -- suggesting a single
# network that takes (theta, p, log t) as input and outputs F directly
# should be able to learn that surface without needing an intermediate
# distributional-family assumption at all.
#
# For each param set: imagine drawing the percentile vs BKT curve for each network
# and putting 1s below the curve and 0s above the curve. Average over all networks.
#
# Training data: for each (theta,p) group (same p_vals grid used
# elsewhere), the empirical CDF is exact at the trials' own order
# statistics -- (log10(t_i), ecdf_i) for i=1..n, ecdf_i=(i-0.5)/n -- no
# parametric fitting step in between. Flatten ALL of these (theta,p,log t)
# -> ecdf rows, across every usable group, into one big supervised
# regression dataset. Each group's n order statistics are highly redundant
# (adjacent sorted values differ by very little in ecdf), so each group is
# subsampled down to at most MAX_PER_GROUP evenly-spaced-by-RANK points
# (preserves the full ecdf range 0->1 and the tail behavior, just drops
# redundant middle density) to keep the flattened dataset a manageable size
# for training.
p_vals = np.array([0.001, 0.002, 0.005, 0.01, 0.02, 0.05, 0.1, 0.15, 0.2, 0.3,
                    0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95])
MIN_SAMPLES = 20
MAX_PER_GROUP = 80


def group_ecdf(ps_idx, p):
    """Empirical (log10(t), ecdf) order-statistic pairs for one (theta,p)
    group -- the raw ingredient both this cell's flattened training set and
    the cross-model sse comparison cell are built from, factored out here so
    that shared logic (pull per-trial percentile-BKT values, drop Nones,
    apply MIN_SAMPLES, sort, compute plotting-position ecdf) lives in one
    place instead of two. Returns None if too few samples; otherwise
    (log_vals, ecdf), both length n, FULL (not yet subsampled) -- callers
    that want MAX_PER_GROUP-subsampled rows (like the training-data loop
    below) subsample after calling this; callers that want the full
    empirical curve (like the sse comparison) use it as-is."""
    ps = all_param_sets[ps_idx]
    vals = [quantile_bkt(bkt, p, Np) for bkt in ps['trial_data']]  # each bkt is a 1D array of t values; then vals[network k] is the pth percentile BKT for network k, right?
    vals = np.array([v for v in vals if v is not None])
    if len(vals) < MIN_SAMPLES:
        return None
    log_vals = np.sort(np.log10(vals))
    n = len(log_vals)  # the number of networks that had a non-None pth percentile BKT
    ecdf = (np.arange(1, n + 1) - 0.5) / n  # so the lowest empirical pth-percentile BKT displays a probability of 0.5/#networks having attained p% by that BKT, the highest gets a probability of (n-.5)/n
    return log_vals, ecdf


rows = []
for ps_idx in USABLE_PS:
    disc, lr, b = recover_theta(weights[ps_idx])
    for p in p_vals:
        result = group_ecdf(ps_idx, p)
        if result is None:
            continue
        log_vals, ecdf = result
        n = len(log_vals)
        if n > MAX_PER_GROUP:
            keep = np.linspace(0, n - 1, MAX_PER_GROUP).round().astype(int)
            keep = np.unique(keep)
            log_vals, ecdf = log_vals[keep], ecdf[keep]
        for lt, f in zip(log_vals, ecdf):
            rows.append((ps_idx, disc, lr, b, p, lt, f))

flat_df = pd.DataFrame(rows, columns=['paramset_idx', 'discspacing', 'length_ratio', 'b', 'p', 'logt', 'ecdf'])
print(f"Flattened dataset: {len(flat_df)} rows from {flat_df['paramset_idx'].nunique()} param sets "
      f"x up to {len(p_vals)} p_vals x up to {MAX_PER_GROUP} order statistics/group")

In [ ]:
# Inputs: (discspacing, length_ratio, b, logit(p), log10(t)) -- FIVE
# continuous inputs this time (vs. the 4 used in the sibling notebooks,
# which only needed theta+p since t was handled analytically by whichever
# parametric CDF family). Output: a single scalar F in [0,1] via a sigmoid
# last layer (guarantees a valid-range CDF value for any input, same spirit
# as the constrained-output-transform trick used elsewhere, just architecturally
# built in this time instead of applied post-hoc).
#
# Train/val split is BY PARAM SET, not by row -- a lesson learned the hard
# way in the skew-logistic notebook's monotonicity experiment: a row-random
# split lets nearly every theta appear in both train and val (just at
# different p/log t), which is an easier "interpolate for an already-seen
# theta" task, not genuine generalization to new theta. Splitting by whole
# param sets is the honest test, and comparable to how the monotonicity
# experiment's model_nopenalty/model_mono were evaluated.
FEATURE_COLS = ['discspacing', 'length_ratio', 'b', 'p', 'logt']

X_all = np.column_stack([
    flat_df['discspacing'].values, flat_df['length_ratio'].values, flat_df['b'].values,
    logit(flat_df['p'].values), flat_df['logt'].values,
]).astype(np.float64)
y_all = flat_df['ecdf'].values.astype(np.float64)

X_mean, X_std = X_all.mean(axis=0), X_all.std(axis=0)


def normalize_X(X):
    return (X - X_mean) / X_std


X_norm = normalize_X(X_all).astype(np.float32)
y_all = y_all.astype(np.float32)

rng = np.random.default_rng(42)
all_ps = flat_df['paramset_idx'].unique()
val_ps = set(rng.choice(all_ps, size=max(1, int(0.2 * len(all_ps))), replace=False))
is_val = flat_df['paramset_idx'].isin(val_ps).values

X_train, y_train = X_norm[~is_val], y_all[~is_val]
X_val, y_val = X_norm[is_val], y_all[is_val]
print(f"train: {len(X_train)} rows ({(~is_val).sum() and flat_df.loc[~is_val, 'paramset_idx'].nunique()} param sets)  "
      f"val: {len(X_val)} rows ({flat_df.loc[is_val, 'paramset_idx'].nunique()} param sets)")


def build_model():
    l2_reg = 9e-5
    return keras.Sequential([
        keras.layers.Input(shape=(5,)),
        keras.layers.Dense(64, activation="relu", kernel_regularizer=keras.regularizers.l2(l2_reg)),
        keras.layers.Dense(64, activation="relu", kernel_regularizer=keras.regularizers.l2(l2_reg)),
        keras.layers.Dense(64, activation="relu", kernel_regularizer=keras.regularizers.l2(l2_reg)),
        keras.layers.Dense(64, activation="relu", kernel_regularizer=keras.regularizers.l2(l2_reg)),
        keras.layers.Dense(1, activation="sigmoid"),
    ])


def lr_by_epoch(epoch):
    if epoch < 80:
        return 1e-3
    elif epoch < 120:
        return 5e-4
    else:
        return 1e-4

In [ ]:
# --- Unconstrained baseline: plain MSE regression, no physics penalty ---
model_plain = build_model()
model_plain.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")
t0 = time.time()
history_plain = model_plain.fit(
    X_train, y_train, validation_data=(X_val, y_val), epochs=150, batch_size=512,
    callbacks=[keras.callbacks.LearningRateScheduler(lr_by_epoch, verbose=0)], verbose=0)
print(f"[{time.time()-t0:.0f}s] plain model -- final train loss: {history_plain.history['loss'][-1]:.5f}  "
      f"val loss: {history_plain.history['val_loss'][-1]:.5f}")
model_plain.save(transport_dir / 'direct_cdf_surface_model_plain.keras')

In [ ]:
# --- Physics-constrained version: same data/split/architecture, plus two
# soft penalties enforced via autodiff on the network's own inputs
# (double-backprop / "collocation" style, as in physics-informed NNs):
#
#   dF/d(log t) >= 0   -- F must be non-decreasing in t (valid CDF)
#   dF/dp       <= 0   -- higher-percentile BKT distributions lie to the
#                         right of lower-percentile ones (physical ordering
#                         of order statistics), matching the corrected sign
#                         convention established in the skew-logistic
#                         notebook's monotonicity experiment: for p_i<p_j,
#                         F_{p_i}(t) >= F_{p_j}(t) everywhere.
#
# Both derivatives are taken directly on the Z-SCORED input columns (logit(p)
# and log t, both already z-scored into columns 3 and 4 of X_train) rather
# than converting back to physical units first -- unnecessary extra work,
# since z-scoring (x-mean)/std is a strictly increasing function of x for
# std>0, so the SIGN of dF/d(normalized column) is identical to the sign of
# dF/d(physical quantity). Only the sign matters for a hinge penalty.
#
# Unlike the skew-logistic notebook's monotonicity experiment -- which had
# to batch by param set and compare ADJACENT p rows within a batch, because
# that model only ever produced a CDF via a fitted parametric family evaluated
# at a handful of discrete p's -- this network takes (p, log t) as direct
# continuous inputs, so the constraint can be checked POINTWISE at each
# training row via its own input gradient. No special batching needed: plain
# row-shuffled minibatches work fine, exactly like the unconstrained baseline.
LAM_T = 1.0
LAM_P = 1.0
IDX_P, IDX_LOGT = 3, 4  # columns of X_train corresponding to logit(p), logt (both z-scored)


@tf.function
def train_step(model, opt, X_batch, y_batch):
    with tf.GradientTape() as outer_tape:
        with tf.GradientTape() as inner_tape:
            inner_tape.watch(X_batch)
            F_pred = model(X_batch, training=True)
        dF_dX = inner_tape.gradient(F_pred, X_batch)  # (batch, 5), normalized-input-space partials

        data_loss = tf.reduce_mean(tf.square(y_batch - tf.squeeze(F_pred, axis=1)))
        penalty_t = tf.reduce_mean(tf.nn.relu(-dF_dX[:, IDX_LOGT]))  # want dF/d(logt) >= 0
        penalty_p = tf.reduce_mean(tf.nn.relu(dF_dX[:, IDX_P]))      # want dF/d(logit p) <= 0
        loss = data_loss + LAM_T * penalty_t + LAM_P * penalty_p

    grads = outer_tape.gradient(loss, model.trainable_variables)
    opt.apply_gradients(zip(grads, model.trainable_variables))
    return data_loss, penalty_t, penalty_p


def train_constrained(X_train, y_train, X_val, y_val, epochs=150, batch_size=512, verbose=0):
    model = build_model()
    opt = keras.optimizers.Adam(1e-3)
    n = len(X_train)
    history = {'loss': [], 'val_loss': [], 'penalty_t': [], 'penalty_p': []}
    rng_local = np.random.default_rng(1)

    for epoch in range(epochs):
        opt.learning_rate.assign(lr_by_epoch(epoch))
        order = rng_local.permutation(n)
        ep_loss, ep_pt, ep_pp = [], [], []
        for i in range(0, n, batch_size):
            idx = order[i:i + batch_size]
            Xb = tf.constant(X_train[idx])
            yb = tf.constant(y_train[idx])
            dl, pt, pp = train_step(model, opt, Xb, yb)
            ep_loss.append(float(dl)); ep_pt.append(float(pt)); ep_pp.append(float(pp))

        val_pred = model(X_val, training=False).numpy().squeeze(axis=1)
        val_loss = float(np.mean((y_val - val_pred) ** 2))
        history['loss'].append(np.mean(ep_loss))
        history['val_loss'].append(val_loss)
        history['penalty_t'].append(np.mean(ep_pt))
        history['penalty_p'].append(np.mean(ep_pp))
        if verbose and epoch % 25 == 0:
            print(f"epoch {epoch}: loss={history['loss'][-1]:.5f} val_loss={val_loss:.5f} "
                  f"pen_t={history['penalty_t'][-1]:.5f} pen_p={history['penalty_p'][-1]:.5f}")

    return model, history


t0 = time.time()
model_constrained, history_constrained = train_constrained(X_train, y_train, X_val, y_val, epochs=150, verbose=0)
print(f"[{time.time()-t0:.0f}s] constrained model -- final train loss: {history_constrained['loss'][-1]:.5f}  "
      f"val loss: {history_constrained['val_loss'][-1]:.5f}  "
      f"penalty_t: {history_constrained['penalty_t'][-1]:.5f}  penalty_p: {history_constrained['penalty_p'][-1]:.5f}")
model_constrained.save(transport_dir / 'direct_cdf_surface_model_constrained.keras')

In [ ]:
# --- Cross-model comparison metric: same "CDF-space sse against the FULL
# (non-subsampled) empirical order statistics per group" reported in all
# three sibling notebooks -- lets this approach's fit quality be compared
# apples-to-apples against the PDF/MLE baseline (mean sse 0.1015), the
# CDF/2-erf mixture (0.0386), and the CDF/skew-logistic (0.0928) models.
def group_sse(predict_fn):
    sses = []
    for ps_idx in USABLE_PS:
        disc, lr, b = recover_theta(weights[ps_idx])
        for p in p_vals:
            result = group_ecdf(ps_idx, p)  # full (non-subsampled) order statistics -- see cell-4
            if result is None:
                continue
            log_vals, ecdf = result
            n = len(log_vals)
            X_g = normalize_X(np.column_stack([
                np.full(n, disc), np.full(n, lr), np.full(n, b), np.full(n, logit(p)), log_vals,
            ])).astype(np.float32)
            F_pred = predict_fn(X_g)
            sses.append(np.sum((F_pred - ecdf) ** 2))  # same convention as sibling notebooks: sum of squared residuals over that group's own (full, non-subsampled) points
    return np.array(sses)


sse_plain = group_sse(lambda X: model_plain.predict(X, verbose=0).squeeze(axis=1))
sse_constrained = group_sse(lambda X: model_constrained(X, training=False).numpy().squeeze(axis=1))

print("Direct-CDF-surface NN, PLAIN (unconstrained):       sse mean=%.4f  median=%.4f  max=%.4f" %
      (sse_plain.mean(), np.median(sse_plain), sse_plain.max()))
print("Direct-CDF-surface NN, CONSTRAINED (physics penalty): sse mean=%.4f  median=%.4f  max=%.4f" %
      (sse_constrained.mean(), np.median(sse_constrained), sse_constrained.max()))
print("\nFor reference (from the sibling notebooks' own conclusion cells):")
print("  PDF/MLE baseline (2-Gaussian mixture, MLE):      sse mean=0.1015")
print("  CDF/2-erf mixture (2-Gaussian mixture, CDF-LS):  sse mean=0.0386")
print("  CDF/skew-logistic (single S-curve, CDF-LS):      sse mean=0.0928")

In [ ]:
# --- Monotonicity violation check on a fine grid, for a few representative
# param sets -- same style of check as the skew-logistic notebook's
# before/after comparison, but here checking the ACTUAL constraint the
# training penalty targets (sign of finite-difference slopes in p and in
# log t across a fine grid), for both the plain and constrained models.
n_ps = len(ps_by_p0)
REP_PS_IDX = [ps_by_p0[0], ps_by_p0[n_ps // 4], ps_by_p0[n_ps // 2],
              ps_by_p0[3 * n_ps // 4], ps_by_p0[-1]]
P_GRID = np.linspace(0.02, 0.95, 60)


def compute_grid(predict_fn, ps_idx):
    ps = all_param_sets[ps_idx]
    disc, lr, b = recover_theta(weights[ps_idx])
    all_logvals = []
    for p in [0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95]:
        vals = [quantile_bkt(bkt, p, Np) for bkt in ps['trial_data']]
        vals = np.array([v for v in vals if v is not None])
        if len(vals):
            all_logvals.append(np.log10(vals))
    all_logvals = np.concatenate(all_logvals)
    t_lo, t_hi = all_logvals.min() - 0.3, all_logvals.max() + 0.3
    T_GRID = np.linspace(t_lo, t_hi, 150)

    Tm, Pm = np.meshgrid(T_GRID, P_GRID)  # both (n_p, n_t)
    X_grid = normalize_X(np.column_stack([
        np.full(Tm.size, disc), np.full(Tm.size, lr), np.full(Tm.size, b),
        logit(Pm.ravel()), Tm.ravel(),
    ])).astype(np.float32)
    Z = predict_fn(X_grid).reshape(Tm.shape)
    return T_GRID, Z


def violation_rates(predict_fn):
    frac_t_viol, frac_p_viol = [], []
    for ps_idx in REP_PS_IDX:
        T_GRID, Z = compute_grid(predict_fn, ps_idx)
        dZ_dt = np.diff(Z, axis=1)   # along log t (axis=1) -- want >= 0
        dZ_dp = np.diff(Z, axis=0)   # along p (axis=0) -- want <= 0
        frac_t_viol.append((dZ_dt < -1e-6).mean())
        frac_p_viol.append((dZ_dp > 1e-6).mean())
    return np.mean(frac_t_viol), np.mean(frac_p_viol)


t_viol_plain, p_viol_plain = violation_rates(lambda X: model_plain.predict(X, verbose=0).squeeze(axis=1))
t_viol_cons, p_viol_cons = violation_rates(lambda X: model_constrained(X, training=False).numpy().squeeze(axis=1))

print("Fraction of adjacent grid-cell steps violating each constraint (avg over 5 representative param sets):")
print(f"  PLAIN:       dF/d(logt)<0 (should be >=0): {100*t_viol_plain:.2f}%   "
      f"dF/dp>0 (should be <=0): {100*p_viol_plain:.2f}%")
print(f"  CONSTRAINED: dF/d(logt)<0 (should be >=0): {100*t_viol_cons:.2f}%   "
      f"dF/dp>0 (should be <=0): {100*p_viol_cons:.2f}%")

In [ ]:
# --- Sibling-model contexts: PDF/MLE baseline, CDF-erf mixture, and
# CDF-logistic, rebuilt here (precompute + normalization + load already-
# trained model) so their surfaces can be plotted alongside the empirical
# data and this notebook's own direct-surface NNs below. No new NN training
# happens in this cell -- each sibling model is already trained and saved by
# its own notebook ("[per p] distrib of pth percentile bkt, PDF fit - GMM
# and NN.ipynb" / "..., CDF fit - ERF and NN.ipynb" / "..., CDF fit -
# logistic and NN.ipynb"); only the per-group parametric fits needed to
# reconstruct each model's exact training-time normalization
# (X_mean/X_std/Y_mean/Y_std, never saved alongside the .keras file itself)
# are recomputed here, using the identical fit functions from those
# notebooks (kept in sync manually, same convention as this project's other
# copy-pasted-with-attribution shared code).
def erf_mixture_cdf(x, w1, mu1, sigma1, mu2, sigma2):
    z1 = (x - mu1) / (sigma1 * np.sqrt(2))
    z2 = (x - mu2) / (sigma2 * np.sqrt(2))
    return w1 * 0.5 * (1 + erf(z1)) + (1 - w1) * 0.5 * (1 + erf(z2))


def genlogistic_cdf(x, mu, s, nu):
    z = (x - mu) / s
    return np.exp(-nu * np.logaddexp(0.0, -z))


def fit_gaussian_mixture_cdf(vals, n_init=5, seed=0):
    """PDF/MLE fit (2-component Gaussian mixture via sklearn EM) -- see
    "..., PDF fit - GMM and NN.ipynb" cell-5 for the full rationale."""
    x = np.log10(np.asarray(vals, dtype=float))
    x = x[np.isfinite(x)]
    if len(x) < 5:
        return None
    gm = GaussianMixture(n_components=2, random_state=seed, n_init=n_init, reg_covar=1e-6).fit(x.reshape(-1, 1))
    order = np.argsort(gm.means_.flatten())
    w = gm.weights_[order]
    mu = gm.means_.flatten()[order]
    sigma = np.sqrt(gm.covariances_.flatten()[order])
    w1, mu1, sigma1, mu2, sigma2 = w[0], mu[0], sigma[0], mu[1], sigma[1]
    xs = np.sort(x)
    n = len(xs)
    ecdf = (np.arange(1, n + 1) - 0.5) / n
    sse = np.sum((erf_mixture_cdf(xs, w1, mu1, sigma1, mu2, sigma2) - ecdf) ** 2)
    return {'w1': w1, 'mu1': mu1, 'sigma1': sigma1, 'mu2': mu2, 'sigma2': sigma2, 'sse': sse}


def fit_erf_mixture_cdf(vals, n_init=5, seed=0):
    """Direct CDF-least-squares fit (same 2-Gaussian-mixture family as
    fit_gaussian_mixture_cdf) -- see "..., CDF fit - ERF and NN.ipynb"
    cell-5 for the full rationale, including why mu/sigma are bounded."""
    x = np.sort(np.log10(np.asarray(vals, dtype=float)))
    x = x[np.isfinite(x)]
    n = len(x)
    if n < 5:
        return None
    ecdf = (np.arange(1, n + 1) - 0.5) / n
    data_lo, data_hi = x[0], x[-1]
    data_range = max(data_hi - data_lo, 1e-3)
    mu_lo, mu_hi = data_lo - 3 * data_range, data_hi + 3 * data_range
    sigma_hi = 5 * data_range
    rng = np.random.default_rng(seed)

    def residuals(params):
        w1, mu1, sigma1, mu2, sigma2 = params
        return erf_mixture_cdf(x, w1, mu1, sigma1, mu2, sigma2) - ecdf

    best = None
    for _ in range(n_init):
        q = rng.uniform(0.15, 0.85)
        split = np.quantile(x, q)
        lo, hi = x[x <= split], x[x > split]
        mu1_0 = lo.mean() if len(lo) else x.mean()
        mu2_0 = hi.mean() if len(hi) else x.mean()
        sigma1_0 = max(lo.std(), 1e-3) if len(lo) else 1.0
        sigma2_0 = max(hi.std(), 1e-3) if len(hi) else 1.0
        w0 = np.clip(len(lo) / n, 0.05, 0.95)
        x0 = [w0, mu1_0, sigma1_0, mu2_0, sigma2_0]
        try:
            res = least_squares(residuals, x0,
                                 bounds=([0, mu_lo, 1e-6, mu_lo, 1e-6],
                                         [1, mu_hi, sigma_hi, mu_hi, sigma_hi]))
        except Exception:
            continue
        sse = np.sum(res.fun ** 2)
        if best is None or sse < best[0]:
            best = (sse, res.x)
    if best is None:
        return None
    sse, (w1, mu1, sigma1, mu2, sigma2) = best
    if mu1 > mu2:
        w1, mu1, sigma1, mu2, sigma2 = 1 - w1, mu2, sigma2, mu1, sigma1
    return {'w1': w1, 'mu1': mu1, 'sigma1': sigma1, 'mu2': mu2, 'sigma2': sigma2, 'sse': sse}


def fit_genlogistic_cdf(vals, n_init=8, seed=0, nu_hi=20.0):
    """Skew-logistic (Type I generalized logistic) fit -- see "..., CDF fit
    - logistic and NN.ipynb" cell-5 for the full rationale, including why
    nu_hi=20 despite ~42% of real fits hitting that bound."""
    x = np.sort(np.log10(np.asarray(vals, dtype=float)))
    x = x[np.isfinite(x)]
    n = len(x)
    if n < 5:
        return None
    ecdf = (np.arange(1, n + 1) - 0.5) / n
    data_lo, data_hi = x[0], x[-1]
    data_range = max(data_hi - data_lo, 1e-3)
    mu_lo, mu_hi = data_lo - 3 * data_range, data_hi + 3 * data_range
    s_hi = 5 * data_range
    rng = np.random.default_rng(seed)

    def residuals(params):
        mu, s, nu = params
        return genlogistic_cdf(x, mu, s, nu) - ecdf

    best = None
    for _ in range(n_init):
        mu0 = np.median(x) + rng.uniform(-0.3, 0.3) * data_range
        s0 = max(x.std(), 1e-3) * rng.uniform(0.5, 1.5)
        nu0 = rng.uniform(0.3, 3.0)
        x0 = [mu0, s0, nu0]
        try:
            res = least_squares(residuals, x0, bounds=([mu_lo, 1e-6, 1e-3], [mu_hi, s_hi, nu_hi]))
        except Exception:
            continue
        sse = np.sum(res.fun ** 2)
        if best is None or sse < best[0]:
            best = (sse, res.x)
    if best is None:
        return None
    sse, (mu, s, nu) = best
    return {'mu': mu, 's': s, 'nu': nu, 'sse': sse}


def build_mixture_context(fit_fn, model_filename):
    """Shared context-builder for the two 5-parameter (w1,mu1,sigma1,mu2,
    sigma2) erf-mixture models (PDF/MLE baseline and CDF-erf mixture) --
    they differ only in fit_fn (MLE/EM vs. direct CDF least-squares).
    Reuses this notebook's own p_vals/MIN_SAMPLES (cell-4) -- identical
    grid used by both sibling notebooks when they trained these models."""
    records = []
    for ps_idx in USABLE_PS:
        ps = all_param_sets[ps_idx]
        disc, lr, b = recover_theta(weights[ps_idx])
        for p in p_vals:
            vals = [quantile_bkt(bkt, p, Np) for bkt in ps['trial_data']]
            vals = np.array([v for v in vals if v is not None])
            if len(vals) < MIN_SAMPLES:
                continue
            fit = fit_fn(vals)
            if fit is None:
                continue
            records.append({'paramset_idx': ps_idx, 'discspacing': disc, 'length_ratio': lr, 'b': b, 'p': p, **fit})
    fit_df_ctx = pd.DataFrame(records)

    TARGET_COLS = ['w1', 'mu1', 'sigma1', 'mu2', 'sigma2']

    def transform_Y(Y):
        w1, mu1, sigma1, mu2, sigma2 = Y[:, 0], Y[:, 1], Y[:, 2], Y[:, 3], Y[:, 4]
        w1c = np.clip(w1, 1e-4, 1 - 1e-4)
        return np.column_stack([np.log(w1c / (1 - w1c)), mu1, np.log(sigma1), mu2, np.log(sigma2)])

    def inverse_transform_Y(Yt):
        lw1, mu1, lsigma1, mu2, lsigma2 = Yt[:, 0], Yt[:, 1], Yt[:, 2], Yt[:, 3], Yt[:, 4]
        w1 = 1 / (1 + np.exp(-lw1))
        return np.column_stack([w1, mu1, np.exp(lsigma1), mu2, np.exp(lsigma2)])

    X_all_ctx = np.column_stack([fit_df_ctx[['discspacing', 'length_ratio', 'b']].values, logit(fit_df_ctx['p'].values)])
    Y_all_ctx = transform_Y(fit_df_ctx[TARGET_COLS].values)
    Xm, Xs = X_all_ctx.mean(axis=0), X_all_ctx.std(axis=0)
    Ym, Ys = Y_all_ctx.mean(axis=0), Y_all_ctx.std(axis=0)

    def normalize_X_ctx(X):
        return (X - Xm) / Xs

    def denormalize_Y_ctx(Y):
        return inverse_transform_Y(Y * Ys + Ym)

    model_ctx = keras.models.load_model(transport_dir / model_filename, compile=False)
    return {'cdf_fn': erf_mixture_cdf, 'normalize_X': normalize_X_ctx, 'denormalize_Y': denormalize_Y_ctx, 'model': model_ctx}


def build_logistic_context(model_filename):
    """Context-builder for the 3-parameter (mu,s,nu) skew-logistic model."""
    records = []
    for ps_idx in USABLE_PS:
        ps = all_param_sets[ps_idx]
        disc, lr, b = recover_theta(weights[ps_idx])
        for p in p_vals:
            vals = [quantile_bkt(bkt, p, Np) for bkt in ps['trial_data']]
            vals = np.array([v for v in vals if v is not None])
            if len(vals) < MIN_SAMPLES:
                continue
            fit = fit_genlogistic_cdf(vals)
            if fit is None:
                continue
            records.append({'paramset_idx': ps_idx, 'discspacing': disc, 'length_ratio': lr, 'b': b, 'p': p, **fit})
    fit_df_ctx = pd.DataFrame(records)

    TARGET_COLS = ['mu', 's', 'nu']

    def transform_Y(Y):
        mu, s, nu = Y[:, 0], Y[:, 1], Y[:, 2]
        return np.column_stack([mu, np.log(s), np.log(nu)])

    def inverse_transform_Y(Yt):
        mu, ls, lnu = Yt[:, 0], Yt[:, 1], Yt[:, 2]
        return np.column_stack([mu, np.exp(ls), np.exp(lnu)])

    X_all_ctx = np.column_stack([fit_df_ctx[['discspacing', 'length_ratio', 'b']].values, logit(fit_df_ctx['p'].values)])
    Y_all_ctx = transform_Y(fit_df_ctx[TARGET_COLS].values)
    Xm, Xs = X_all_ctx.mean(axis=0), X_all_ctx.std(axis=0)
    Ym, Ys = Y_all_ctx.mean(axis=0), Y_all_ctx.std(axis=0)

    def normalize_X_ctx(X):
        return (X - Xm) / Xs

    def denormalize_Y_ctx(Y):
        return inverse_transform_Y(Y * Ys + Ym)

    model_ctx = keras.models.load_model(transport_dir / model_filename, compile=False)
    return {'cdf_fn': genlogistic_cdf, 'normalize_X': normalize_X_ctx, 'denormalize_Y': denormalize_Y_ctx, 'model': model_ctx}


ctx_pdf_mle = build_mixture_context(fit_gaussian_mixture_cdf, 'percentile_erf_model.keras')
print("Built PDF/MLE baseline context")
ctx_cdf_erf = build_mixture_context(fit_erf_mixture_cdf, 'percentile_cdf_erf_mixture_model.keras')
print("Built CDF-erf-mixture context")
ctx_cdf_logistic = build_logistic_context('percentile_cdf_skewlogistic_model.keras')
print("Built CDF-logistic context")


def compute_grid_parametric(ctx, ps_idx):
    """Same t-grid logic as compute_grid above, but for a 'sibling'
    parametric model: predict this model's own (theta,p)-conditioned
    params ONCE per p (not per (p,logt) grid point, since t only enters
    analytically via cdf_fn), then evaluate the model's own cdf_fn across
    the shared T_GRID."""
    ps = all_param_sets[ps_idx]
    disc, lr, b = recover_theta(weights[ps_idx])
    all_logvals = []
    for p in [0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95]:
        vals = [quantile_bkt(bkt, p, Np) for bkt in ps['trial_data']]
        vals = np.array([v for v in vals if v is not None])
        if len(vals):
            all_logvals.append(np.log10(vals))
    all_logvals = np.concatenate(all_logvals)
    t_lo, t_hi = all_logvals.min() - 0.3, all_logvals.max() + 0.3
    T_GRID = np.linspace(t_lo, t_hi, 150)

    X_grid = np.array([[disc, lr, b, logit(p)] for p in P_GRID])
    Y_pred = ctx['denormalize_Y'](ctx['model'].predict(ctx['normalize_X'](X_grid), verbose=0))
    Z = np.stack([ctx['cdf_fn'](T_GRID, *params) for params in Y_pred], axis=0)
    return T_GRID, Z

In [ ]:
# --- Visual comparison: joint CDF surface, empirical + all 5 models
# (PDF/MLE baseline, CDF/2-erf mixture, CDF/skew-logistic, direct NN plain,
# direct NN constrained), for a couple of representative param sets. Top
# row: 2D pcolormesh+contour (scatter for empirical). Bottom row: the same
# surfaces rendered in 3D (3D scatter for empirical).
#
# Empirical panels: no smooth surface exists for raw data (F is only ever
# known at the discrete p_vals training grid), so instead of a pcolormesh/
# plot_surface they're a scatter of every individual (log t, p[, ecdf])
# empirical order-statistic point, colored by that point's own ecdf value --
# same colormap/scale (viridis, 0-1) as the model panels, so the whole row
# reads as one visually consistent comparison despite this one column being
# real discrete data rather than a continuous predicted field.
CONTOUR_LEVELS = [0.1, 0.3, 0.5, 0.7, 0.9]
COMPARE_PS_IDX = [REP_PS_IDX[0], REP_PS_IDX[-1]]  # best- and worst-connected

MODEL_PANELS = [
    ('PDF / MLE baseline', ctx_pdf_mle),
    ('CDF / 2-erf mixture', ctx_cdf_erf),
    ('CDF / skew-logistic', ctx_cdf_logistic),
    ('direct NN, plain', {'predict_fn': lambda X: model_plain.predict(X, verbose=0).squeeze(axis=1)}),
    ('direct NN, constrained', {'predict_fn': lambda X: model_constrained(X, training=False).numpy().squeeze(axis=1)}),
]


def style_axis_2d(ax, T_GRID):
    tick_lo, tick_hi = int(np.floor(T_GRID.min())), int(np.ceil(T_GRID.max()))
    ticks = np.arange(tick_lo, tick_hi + 1)
    ax.set_xticks(ticks)
    ax.set_xticklabels([f'$10^{{{t:.0f}}}$' for t in ticks])
    ax.set_xlim(T_GRID.min(), T_GRID.max())
    ax.set_xlabel('breakthrough time (s)', fontsize=9)
    ax.set_ylabel('percentile $p$', fontsize=9)
    ax.tick_params(labelsize=8)


def style_axis_3d(ax, T_GRID):
    tick_lo, tick_hi = int(np.floor(T_GRID.min())), int(np.ceil(T_GRID.max()))
    ticks = np.arange(tick_lo, tick_hi + 1)
    ax.set_xticks(ticks)
    ax.set_xticklabels([f'$10^{{{t:.0f}}}$' for t in ticks])
    ax.set_xlabel('log t', labelpad=6, fontsize=8)
    ax.set_ylabel('$p$', labelpad=4, fontsize=8)
    ax.set_zlabel('$F$', labelpad=2, fontsize=8)
    ax.set_zlim(0, 1)
    ax.tick_params(labelsize=6)
    ax.view_init(elev=25, azim=-60)


pdf_path = transport_dir / '[per p, t] probability of >=p percent having bkt <= t, NN plain vs NN constrained.pdf'
with PdfPages(pdf_path) as pdf:
    for ps_idx in COMPARE_PS_IDX:
        ps = all_param_sets[ps_idx]
        disc, lr, b = recover_theta(weights[ps_idx])
        n_cols = 1 + len(MODEL_PANELS)  # empirical + 5 models

        fig = plt.figure(figsize=(5.0 * n_cols, 9.5))

        # --- column 1: empirical (2D scatter on top, 3D scatter below) ---
        ax_emp = fig.add_subplot(2, n_cols, 1)
        ax_emp_3d = fig.add_subplot(2, n_cols, n_cols + 1, projection='3d')
        for p in p_vals:
            result = group_ecdf(ps_idx, p)
            if result is None:
                continue
            log_vals, ecdf = result
            ax_emp.scatter(log_vals, np.full_like(log_vals, p), c=ecdf, cmap='viridis',
                            vmin=0, vmax=1, s=4, linewidths=0)
            ax_emp_3d.scatter(log_vals, np.full_like(log_vals, p), ecdf, c=ecdf, cmap='viridis',
                               vmin=0, vmax=1, s=3, linewidths=0)
        ax_emp.set_ylim(0, 1)
        ax_emp.set_title('empirical', fontsize=11)

        # --- columns 2..6: models (2D pcolormesh+contour, 3D plot_surface) ---
        mesh_for_cbar = None
        T_GRID_ref = None
        for i, (name, panel) in enumerate(MODEL_PANELS):
            col = i + 2  # 1-indexed subplot position; empirical occupies col 1
            if 'predict_fn' in panel:
                T_GRID, Z = compute_grid(panel['predict_fn'], ps_idx)
            else:
                T_GRID, Z = compute_grid_parametric(panel, ps_idx)
            if T_GRID_ref is None:
                T_GRID_ref = T_GRID  # same range for every panel at this ps_idx -- reused for the empirical panel's axis styling below

            ax2d = fig.add_subplot(2, n_cols, col)
            mesh = ax2d.pcolormesh(T_GRID, P_GRID, Z, shading='auto', cmap='viridis', vmin=0, vmax=1)
            cs = ax2d.contour(T_GRID, P_GRID, Z, levels=CONTOUR_LEVELS, colors='white', linewidths=1.0, alpha=0.8)
            ax2d.clabel(cs, fmt='%.1f', fontsize=6, colors='white')
            ax2d.set_title(name, fontsize=11)
            style_axis_2d(ax2d, T_GRID)
            mesh_for_cbar = mesh

            Tmesh, Pmesh = np.meshgrid(T_GRID, P_GRID)
            ax3d = fig.add_subplot(2, n_cols, n_cols + col, projection='3d')
            ax3d.plot_surface(Tmesh, Pmesh, Z, cmap='viridis', vmin=0, vmax=1,
                               rstride=2, cstride=4, linewidth=0, antialiased=True)
            style_axis_3d(ax3d, T_GRID)

        style_axis_2d(ax_emp, T_GRID_ref)
        style_axis_3d(ax_emp_3d, T_GRID_ref)

        fig.colorbar(mesh_for_cbar, ax=fig.get_axes(), pad=0.01, shrink=0.6, label='CDF $F$')
        fig.suptitle(f'Param set {ps_idx}  disc={disc:.2f}, r={lr:.2f}, b={b:.2e}, p0={ps["p0"]:.3f}', fontsize=15)
        pdf.savefig(fig, bbox_inches='tight')
        plt.close(fig)

print(f'Saved {pdf_path}  ({len(COMPARE_PS_IDX)} pages)')

In [ ]:
# --- Conclusion ---
#
# Fit quality (cell-8), sse across all 1479 (theta,p) groups, vs. the three
# sibling parametric-family models:
#   PDF/MLE baseline (2-Gaussian mixture, MLE):          mean 0.1015
#   CDF/2-erf mixture (2-Gaussian mixture, CDF-LS):       mean 0.0386
#   CDF/skew-logistic (single S-curve, CDF-LS):           mean 0.0928
#   Direct surface NN, plain:        mean 3.5354  median 0.1128  max 193.3
#   Direct surface NN, constrained:  mean 4.3565  median 0.0391  max 217.7
# The MEDIAN is genuinely competitive -- constrained's median (0.0391) ties
# the best parametric model (CDF/2-erf mixture, 0.0386), and plain's median
# (0.1128) is in the same range as the PDF/MLE baseline. But the MEAN is
# 30-40x worse than every parametric model, entirely because of a small
# number of catastrophic outliers (max sse ~193-218, vs ~0.2-0.8 for the
# parametric models). Diagnosed directly: EVERY one of the 10 worst rows
# (both models) is the SAME single held-out param set (idx 88) -- its
# predicted F barely moves (stays within [0.003, 0.87]) across the entire
# log-t domain, for every p. This isn't an out-of-range extrapolation
# problem: param set 88's theta (disc=1.29, r=0.40, b=2.3e-5) sits well
# inside the training thetas' range on every coordinate individually. It's
# a genuine single-theta interpolation failure for this particular
# train/val split and network initialization -- plausibly because this
# model has to learn the full 2D (p, log t) -> F SHAPE for held-out theta
# purely by interpolating in a 3D theta neighborhood, a harder generalization
# demand than the parametric-family models face (they only need to
# interpolate 5, or 3, smooth summary numbers per theta). This is the real,
# structural risk of this approach vs. the sibling models -- worth being
# upfront about rather than only reporting the (flattering) median.
#
# Training itself (cell-6/7): both models trained in <30s (150 epochs) on
# the 118,320-row flattened dataset -- an order of magnitude faster than the
# custom monotonicity-penalized loop in the skew-logistic notebook, because
# the constraint here is POINTWISE per training row (no batching-by-param-set
# needed), unlike that notebook's paired-rows-within-a-batch approach.
#
# Monotonicity (cell-9, fine-grid check on 5 representative param sets):
#   PLAIN:       dF/d(logt)<0 violations 0.00%   dF/dp>0 violations 0.07%
#   CONSTRAINED: dF/d(logt)<0 violations 0.00%   dF/dp>0 violations 0.18%
# Both models are already ~monotonic in log t even with NO penalty (0.00%
# both ways) -- unsurprising, since regressing directly onto empirical CDF
# targets (which ARE monotonic in t by construction) gives the network a
# strong implicit bias toward a monotonic-in-t fit, unlike the parametric-
# family models where t-monotonicity was guaranteed analytically by the CDF
# formula itself, not learned. The p-direction is more interesting: the
# CONSTRAINED model shows MORE violations on this grid check (0.18% vs
# 0.07%) despite its own training penalty (cell-7) converging to ~0.00000 by
# the end of training -- i.e. the penalty was satisfied almost everywhere it
# was actually CHECKED (real training-row locations), but this grid check
# also probes many (theta,p,logt) combinations that were never a real
# training row and never had the penalty applied there. This is exactly the
# "no collocation points" limitation noted below, not a contradiction --
# and visually (cell-10's PDF), both models' surfaces for the two inspected
# param sets look clean and smoothly monotonic either way, at least as good
# as the PDF/MLE-baseline notebook's own joint-surface plots for the same
# param sets.
#
# Net takeaway: promising but not yet a clean win. Typical-case fit quality
# (median) already matches the best parametric model, achieved with a
# genuinely simpler idea (no distributional-family assumption at all) and
# faster training -- but the mean is dragged down hard by at least one
# outright generalization failure on a held-out theta, and the physics
# penalty as implemented here (pointwise-only, no collocation points) didn't
# demonstrably improve monotonicity on a held-out fine grid despite driving
# its own training objective to ~0. Both issues point to the same fix:
# broaden where the model/penalty are actually evaluated during training --
# collocation points for the penalty, and likely more training data
# (raise MAX_PER_GROUP, or train on multiple random splits and check
# whether param-set-88-style failures are split-specific or systematic)
# for the outlier problem -- rather than a fundamental flaw in the direct-
# surface-learning idea itself.
#
# Known limitations of this implementation, for whoever extends it:
# - MAX_PER_GROUP=80 subsampling (cell-4) discards most of each group's raw
#   order statistics before flattening -- chosen to keep the flattened
#   dataset a manageable size for training, but means the model never sees
#   the very densest/most granular parts of any single group's empirical
#   CDF; if fit quality (cell-8) is noticeably worse than the parametric-
#   family models, retrying with a higher MAX_PER_GROUP (or no subsampling
#   at all, at the cost of a much larger flattened dataset and slower
#   training) is the first thing to try.
# - The physics penalty (cell-7) is evaluated only at actual TRAINING rows'
#   own (theta,p,logt) locations, not at additional randomly-sampled
#   "collocation points" spanning the full input domain (the more complete
#   version of this technique, as used in physics-informed neural networks)
#   -- meaning regions of (theta,p,logt) space far from any real training
#   point have no enforcement at all, which is very likely why cell-9's
#   grid-based violation check didn't clearly improve. Adding a separate
#   collocation-point sampling step (drawing random theta/p/logt
#   combinations, independent of whether real data exists there, and
#   applying only the penalty terms -- no data_loss term -- at those
#   points) would be a natural next extension, and is standard practice in
#   the physics-informed NN literature for exactly this reason.
# - LAM_T=LAM_P=1.0 (cell-7) were not tuned -- picked as a reasonable
#   starting point, matching the untuned lam=1.0 used in the skew-logistic
#   notebook's monotonicity experiment, not the result of a sweep.
# - Single random train/val split and single network initialization per
#   model (same caveat as every other model-comparison notebook in this
#   project) -- the param-set-88 failure in particular should be checked
#   against at least one more random split before concluding it's a
#   structural weakness of the approach rather than one unlucky run.

In [ ]:
# --- Error score per theta: mean(|pred_F(p,t) - actual_F(p,t)|) over all
# the empirical (p,t) points for that theta (same full, non-subsampled
# points via group_ecdf used elsewhere in this notebook) -- a single number
# summarizing how well model_plain/model_constrained's OWN (already-frozen,
# cells 6-7) predictions matched the real data for that param set. No
# circularity: this is computed AFTER those models are trained, using their
# fixed predictions, so it's an ordinary fixed label for the small
# theta-only error-score model trained in the next cell.
error_rows = []
for ps_idx in USABLE_PS:
    disc, lr, b = recover_theta(weights[ps_idx])
    for p in p_vals:
        result = group_ecdf(ps_idx, p)
        if result is None:
            continue
        log_vals, ecdf = result
        X_g = normalize_X(np.column_stack([
            np.full(len(log_vals), disc), np.full(len(log_vals), lr), np.full(len(log_vals), b),
            np.full(len(log_vals), logit(p)), log_vals,
        ])).astype(np.float32)
        abs_err_plain = np.abs(model_plain.predict(X_g, verbose=0).squeeze(axis=1) - ecdf)
        abs_err_constrained = np.abs(model_constrained(X_g, training=False).numpy().squeeze(axis=1) - ecdf)
        for ae_p, ae_c in zip(abs_err_plain, abs_err_constrained):
            error_rows.append((ps_idx, disc, lr, b, ae_p, ae_c))

error_pt_df = pd.DataFrame(error_rows, columns=['paramset_idx', 'discspacing', 'length_ratio', 'b',
                                                 'abs_err_plain', 'abs_err_constrained'])
error_score_df = error_pt_df.groupby('paramset_idx').agg(
    discspacing=('discspacing', 'first'), length_ratio=('length_ratio', 'first'), b=('b', 'first'),
    error_score_plain=('abs_err_plain', 'mean'), error_score_constrained=('abs_err_constrained', 'mean'),
).reset_index()

print(f"Computed error scores for {len(error_score_df)} param sets")
print("\nerror_score_plain summary:")
print(error_score_df['error_score_plain'].describe())
print("\nerror_score_constrained summary:")
print(error_score_df['error_score_constrained'].describe())

In [ ]:
# --- Small theta -> error-score network(s): input is JUST theta (disc, lr,
# b) -- no p, no logt, since error_score is already aggregated over all
# (p,t) for a given theta. Own 3-dim normalization (separate from
# normalize_X's 5-dim theta+p+logt space, since the input dimensionality
# differs). Softplus output -- error_score is a mean of absolute values,
# always >= 0. Much smaller architecture than model_plain/model_constrained
# (only ~87 rows total here, one per param set, vs. the ~118K-row flattened
# dataset those were trained on) to avoid overfitting.
X_theta_all = np.column_stack([error_score_df['discspacing'].values, error_score_df['length_ratio'].values,
                                error_score_df['b'].values]).astype(np.float64)
Xt_mean, Xt_std = X_theta_all.mean(axis=0), X_theta_all.std(axis=0)


def normalize_X_theta(X):
    return (X - Xt_mean) / Xt_std


X_theta_norm = normalize_X_theta(X_theta_all).astype(np.float32)
# Same theta-level val split as Stage A (reuses the val_ps SET from cell-5).
is_val_theta = error_score_df['paramset_idx'].isin(val_ps).values

X_theta_train, X_theta_val = X_theta_norm[~is_val_theta], X_theta_norm[is_val_theta]
score_plain_train = error_score_df['error_score_plain'].values[~is_val_theta].astype(np.float32)
score_plain_val = error_score_df['error_score_plain'].values[is_val_theta].astype(np.float32)
score_cons_train = error_score_df['error_score_constrained'].values[~is_val_theta].astype(np.float32)
score_cons_val = error_score_df['error_score_constrained'].values[is_val_theta].astype(np.float32)

print(f"theta->error-score dataset: {len(X_theta_train)} train param sets, {len(X_theta_val)} val param sets")


def build_error_score_model():
    l2_reg = 9e-5
    return keras.Sequential([
        keras.layers.Input(shape=(3,)),
        keras.layers.Dense(16, activation="relu", kernel_regularizer=keras.regularizers.l2(l2_reg)),
        keras.layers.Dense(16, activation="relu", kernel_regularizer=keras.regularizers.l2(l2_reg)),
        keras.layers.Dense(1, activation="softplus"),  # non-negative -- error score is a mean absolute value
    ])


def lr_by_epoch_small(epoch):
    if epoch < 150:
        return 1e-3
    elif epoch < 250:
        return 5e-4
    else:
        return 1e-4


t0 = time.time()
error_score_model_plain = build_error_score_model()
error_score_model_plain.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")
hist_sp = error_score_model_plain.fit(X_theta_train, score_plain_train, validation_data=(X_theta_val, score_plain_val),
                                       epochs=300, batch_size=16,
                                       callbacks=[keras.callbacks.LearningRateScheduler(lr_by_epoch_small, verbose=0)], verbose=0)
print(f"[{time.time()-t0:.0f}s] error_score_model_plain -- final train loss: {hist_sp.history['loss'][-1]:.5f}  "
      f"val loss: {hist_sp.history['val_loss'][-1]:.5f}")
error_score_model_plain.save(transport_dir / 'error_score_model_plain.keras')

t0 = time.time()
error_score_model_constrained = build_error_score_model()
error_score_model_constrained.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")
hist_sc = error_score_model_constrained.fit(X_theta_train, score_cons_train, validation_data=(X_theta_val, score_cons_val),
                                             epochs=300, batch_size=16,
                                             callbacks=[keras.callbacks.LearningRateScheduler(lr_by_epoch_small, verbose=0)], verbose=0)
print(f"[{time.time()-t0:.0f}s] error_score_model_constrained -- final train loss: {hist_sc.history['loss'][-1]:.5f}  "
      f"val loss: {hist_sc.history['val_loss'][-1]:.5f}")
error_score_model_constrained.save(transport_dir / 'error_score_model_constrained.keras')

In [ ]:
# --- Diagnostic: true error_score (computed directly from model_plain/
# model_constrained's own residuals) vs. each small error-score model's own
# predicted value, for every usable param set, split by train/held-out-val
# -- the real test of whether theta alone carries a useful signal for
# predicting fit quality, or whether the model just regresses to the mean.
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, name, true_col, pred_model in zip(
    axes, ['plain', 'constrained'],
    ['error_score_plain', 'error_score_constrained'],
    [error_score_model_plain, error_score_model_constrained],
):
    pred = pred_model.predict(X_theta_norm, verbose=0).squeeze(axis=1)
    true = error_score_df[true_col].values
    ax.scatter(true[~is_val_theta], pred[~is_val_theta], s=20, alpha=0.7, label='train param sets')
    ax.scatter(true[is_val_theta], pred[is_val_theta], s=30, alpha=0.9, label='held-out val param sets', marker='^')
    lims = [0, max(true.max(), pred.max()) * 1.05]
    ax.plot(lims, lims, 'k--', lw=1, alpha=0.5, label='y=x')
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel('true error score (mean |pred_F - actual_F|)')
    ax.set_ylabel('predicted error score')
    ax.set_title(name, fontsize=12)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
fig_path = transport_dir / '[per p, t] probability of >=p percent having bkt <= t, error score true vs predicted.pdf'
plt.savefig(fig_path, dpi=130, bbox_inches='tight')
plt.show()
print(f'Saved {fig_path}')

In [ ]:
# --- Addendum: theta -> error-score models ---
#
# Run against the OG (non-narrowed) param-range dataset (cell-1/cell-2, see
# their comments) -- 60/100 param sets usable, 48 train / 12 held-out val.
#
# True error_score distributions (both models, from the base model_plain/
# model_constrained's own residuals, computed BEFORE either error-score
# model exists -- see cell-71aaab88): both are right-skewed, dominated by a
# tight low-error cluster (median 0.013 plain, 0.007 constrained) with a
# handful of much-worse outlier param sets pulling the mean up (mean 0.036
# plain, 0.038 constrained, max ~0.40 for both) -- i.e. most param sets fit
# well, a few fit badly, consistent with the outlier-driven sse pattern
# already seen for the narrowed dataset in the main conclusion above.
# error_score_constrained's median (0.007) is noticeably BETTER than
# error_score_plain's (0.013), suggesting the physics-informed penalty
# helps typical-case fit quality here even more than it did on the narrowed
# dataset (main conclusion: comparable/slightly worse median sse).
#
# Sanity check -- correlation between the TRUE error score and connectivity
# (p0): corr(p0, error_score_plain) = -0.16, corr(p0, error_score_constrained)
# = -0.15. Both weak and, notably, the OPPOSITE sign from the naive
# expectation (worse-connected param sets = harder to fit = higher error).
# With only 60 param sets and p0 already restricted to <0.9, this suggests
# fit difficulty here isn't simply/mainly driven by connectivity -- probably
# more a function of how spread out or oddly-shaped a param set's BKT
# distribution is, which p0 alone doesn't capture. Worth keeping in mind:
# the theta->error-score model below has no obvious single "easy" feature
# (like p0) to lean on.
#
# theta -> error-score models: both trained to a low train loss (0.003
# plain, 0.002-0.003 constrained) -- the real question is whether that
# generalizes. Pearson/Spearman correlation, true vs. predicted error score:
#   plain:        TRAIN pearson=0.13  spearman=0.32   VAL pearson=0.29  spearman=0.59
#   constrained:   TRAIN pearson=-0.28 spearman=-0.23  VAL pearson=-0.20 spearman=0.00
# Mixed, and genuinely different between the two models -- reported
# honestly rather than averaged together:
# - PLAIN's error-score model shows a real, if modest, positive rank
#   correlation on the 12 held-out param sets (spearman 0.59) -- it can
#   partially rank-order unseen param sets by expected fit difficulty, even
#   though the true-vs-predicted scatter plot (above) shows it compresses
#   the SCALE heavily (predictions mostly stay under ~0.1 even for true
#   scores approaching 0.4) -- it's catching relative difficulty better
#   than absolute magnitude.
# - CONSTRAINED's error-score model shows NO reliable relationship at all
#   -- slightly negative on both train and held-out val, spearman exactly
#   0.00 on val. Given the true error_score_constrained values are both
#   smaller in magnitude AND apparently less related to any theta-only
#   signal available here (harder problem: predicting a rarer, more
#   idiosyncratic set of "bad" param sets from just 3 physical parameters
#   and 48 training examples), this isn't too surprising, but it does mean
#   this particular auxiliary model isn't useful as built.
#
# Net honest takeaway: theta alone carries SOME signal about the direct
# surface NN's own eventual fit quality (clearest for the plain model, via
# rank correlation, not exact magnitude), but this is a weak, noisy
# relationship on a small (60 param set) dataset -- not a reliable,
# deployable "confidence estimate" yet. The constrained model's error score
# in particular doesn't show a usable relationship with theta given this
# implementation. Bigger training set (more param sets) and/or richer
# features than just the 3 raw theta values (e.g. distance-to-nearest-
# training-theta, or an ensemble-disagreement signal) would be the natural
# next things to try if this direction is pursued further.